# Subsystem 1 — Allocation Notebook (Category → Product)
This notebook:
1. Loads trained **Subsystem 1 artifacts** (`subsystem1_artifacts.pkl`) containing:
   - `model` (sklearn Pipeline)
   - `threshold` (tuned decision threshold)
   - `feature_cols` (expected feature columns for inference)
2. Loads **raw transactional data** (your original dataset).
3. Builds a **latest feature snapshot** (per Category) using the most recent window.
4. Predicts `probability_up` per Category and derives `growth_signal`.
5. Computes **product share** per Category from recent history.
6. Allocates category-level forecast budget to **product-level recommended quantities**.

_Last updated: 2026-02-12 12:50_


In [12]:
import os
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Tuple

# Files
#ARTIFACTS_PATH = "subsystem1_artifacts.pkl"   # produced by previous version training notebook
ARTIFACTS_PATH = "models/subsystem1_artifacts_v2.pkl"
#RAW_DATA_PATH  = "ML-Dataset.csv"             # raw dataset
RAW_DATA_PATH = "updated_base_history.csv"

# Prefer feedback-updated base history if it exists; otherwise fall back to original raw dataset.
RAW_DATA_CANDIDATES = [
    "updated_base_history.csv",
    "ML-Dataset.csv",
]
RAW_DATA_PATH = next((p for p in RAW_DATA_CANDIDATES if Path(p).exists()), None)

if RAW_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find a raw/base dataset. Checked: "
        + ", ".join(RAW_DATA_CANDIDATES)
    )

# Windows
FEATURE_WINDOW_DAYS = 28
SHARE_WINDOW_DAYS = 90
FORECAST_HORIZON_DAYS = 28

# Allocation sensitivity
ALPHA = 0.5

# Sanity
print("CWD:", os.getcwd())
print("Chosen RAW_DATA_PATH:", RAW_DATA_PATH)
print("ARTIFACTS_PATH:", ARTIFACTS_PATH)
print("Files:", [f for f in os.listdir() if f.endswith(('.pkl', '.csv', '.ipynb'))])


CWD: /Users/ayemyatmyathmwe/Documents/Murdoch Singapore/BIT_AIAS&BIS_Murdoch/TJA_2026/ICT304/Assignments
Chosen RAW_DATA_PATH: updated_base_history.csv
ARTIFACTS_PATH: models/subsystem1_artifacts_v2.pkl
Files: ['category_social_trends.csv', 'sub1_allocation_v1.ipynb', 'Subsystem1.ipynb', 'integrated_recommendations.csv', 'recommendation_sub4.ipynb', 'Subsystem_2_updated_integrated.ipynb', 'subsystem1_category_output.csv', 'RiskScoringSub.ipynb', 'subsystem4_final_recommendations.csv', 'subsystem1_product_recommendations.csv', 'rolling_window_builder_updated.ipynb', 'ML-Dataset.csv', 'rolling_supervised_dataset.csv', 'updated_base_history.csv', 'Subsystem_2 1.ipynb', 'subsystem1_product_recommendations_v2.csv', 'SubSystem1_v1.ipynb', 'sub1_category_v3.ipynb', 'sub1_category_v3_updated.ipynb', 'subsystem1_artifacts.pkl', 'sub1_category.ipynb', 'sub1_allocation_v1_updated.ipynb', 'subsystem1_best_model.pkl', 'subsystem1_category_output_v2.csv', 'social_trend_signals.csv', 'SubSystem_v2.ip

## 1) Load Subsystem 1 artifacts (model + threshold + feature columns)

In [13]:
artifacts = joblib.load(ARTIFACTS_PATH)
model = artifacts["model"]
BEST_THRESHOLD = float(artifacts["threshold"])
FEATURE_COLS = list(artifacts.get("feature_cols", []))

print("Loaded artifacts:")
print(" - threshold:", BEST_THRESHOLD)
print(" - n_features expected:", len(FEATURE_COLS))
print(" - feature_cols:", FEATURE_COLS)


Loaded artifacts:
 - threshold: 0.1
 - n_features expected: 0
 - feature_cols: []


## 2) Load raw data and basic cleaning

In [14]:
df_raw = pd.read_csv(RAW_DATA_PATH)

# Parse dates
df_raw["OrderDate"] = pd.to_datetime(df_raw["OrderDate"], errors="coerce")

# Basic filters 
if "Status" in df_raw.columns:
    df_raw = df_raw[df_raw["Status"].astype(str).str.lower().eq("shipped")]

# Required columns check
required = {"OrderDate","CategoryName","ProductName","OrderItemQuantity"}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f"Missing required columns in raw dataset: {missing}")

# Ensure numeric quantity
df_raw["OrderItemQuantity"] = pd.to_numeric(df_raw["OrderItemQuantity"], errors="coerce").fillna(0)

print("Raw rows:", len(df_raw))
print("Date range:", df_raw["OrderDate"].min(), "→", df_raw["OrderDate"].max())
print("Categories:", df_raw["CategoryName"].nunique(), "Products:", df_raw["ProductName"].nunique())

Raw rows: 184
Date range: 2013-06-21 00:00:00 → 2017-11-02 00:00:00
Categories: 5 Products: 147


## 3) Build latest category-level feature snapshot (matching training feature columns)
We compute features using the most recent `FEATURE_WINDOW_DAYS` days.

**Note**: This snapshot is for inference, not rolling dataset building.
We also add safe time features: `month`, `week_of_year`, `day_of_week` (from window_end).

In [15]:
def build_latest_category_features(
    df: pd.DataFrame,
    window_days: int = 28
) -> Tuple[pd.DataFrame, pd.Timestamp, pd.Timestamp]:
    """Return per-category features over the most recent window_days."""
    latest_date = df["OrderDate"].max().normalize()
    window_start = latest_date - pd.Timedelta(days=window_days-1)
    window_end = latest_date

    df_recent = df[(df["OrderDate"] >= window_start) & (df["OrderDate"] <= window_end)].copy()

    # Aggregate daily demand per category
    daily = (df_recent
             .groupby(["CategoryName", "OrderDate"])["OrderItemQuantity"]
             .sum()
             .reset_index())

    categories = daily["CategoryName"].unique()
    rows = []

    # Build a full daily index for each category to get correct zeros
    full_idx = pd.date_range(window_start, window_end, freq="D")

    for cat in categories:
        s = daily[daily["CategoryName"] == cat].set_index("OrderDate")["OrderItemQuantity"]
        s = s.reindex(full_idx, fill_value=0)

        cur_mean = float(s.mean())
        cur_std = float(s.std(ddof=0))  # population std
        coverage = float((s > 0).mean())
        volatility_ratio = float(cur_std / (cur_mean + 1e-6))

        # For inference, intra_growth may not be available; set to 0 (neutral)
        intra_growth = 0.0

        # Safe calendar features from window_end
        month = int(window_end.month)
        week_of_year = int(window_end.isocalendar().week)
        day_of_week = int(window_end.dayofweek)

        rows.append({
            "CategoryName": cat,
            "cur_mean": cur_mean,
            "cur_std": cur_std,
            "coverage": coverage,
            "volatility_ratio": volatility_ratio,
            "intra_growth": intra_growth,
            "month": month,
            "week_of_year": week_of_year,
            "day_of_week": day_of_week,
            "window_start": window_start,
            "window_end": window_end
        })

    feat = pd.DataFrame(rows)
    return feat, window_start, window_end

latest_feat, window_start, window_end = build_latest_category_features(df_raw, FEATURE_WINDOW_DAYS)
print("Latest feature snapshot window:", window_start.date(), "→", window_end.date())
latest_feat.head()

Latest feature snapshot window: 2017-10-06 → 2017-11-02


,CategoryName,cur_mean,cur_std,coverage,volatility_ratio,intra_growth,month,week_of_year,day_of_week,window_start,window_end
0,CPU,5.178571,26.908646,0.035714,5.196151,0.0,11,44,3,2017-10-06,2017-11-02
1,Mother Board,2.785714,14.474996,0.035714,5.196151,0.0,11,44,3,2017-10-06,2017-11-02
2,Storage,4.285714,22.269225,0.035714,5.196151,0.0,11,44,3,2017-10-06,2017-11-02
3,Video Card,11.642857,36.641326,0.107143,3.147107,0.0,11,44,3,2017-10-06,2017-11-02


## 4) Align features to training schema and predict category probabilities
We ensure the columns match what the model expects (`FEATURE_COLS`).

In [16]:
if not FEATURE_COLS:
    FEATURE_COLS = [c for c in latest_feat.columns if c not in ["window_start","window_end"]]
    print("No feature_cols found in artifacts; using inferred FEATURE_COLS:", FEATURE_COLS)

# Build X_infer with exactly the expected columns (extra columns dropped, missing filled)
X_infer = latest_feat.copy()

# Remove meta date fields from inference features if present in FEATURE_COLS 
# Keep only FEATURE_COLS in exact order
for col in FEATURE_COLS:
    if col not in X_infer.columns:
        X_infer[col] = 0

X_infer = X_infer[FEATURE_COLS]

# Predict
proba = model.predict_proba(X_infer)[:, 1]
latest_feat["probability_up"] = proba
latest_feat["growth_signal"] = (latest_feat["probability_up"] >= BEST_THRESHOLD).astype(int)

latest_feat[["CategoryName","probability_up","growth_signal","window_start","window_end"]].sort_values("probability_up", ascending=False)

No feature_cols found in artifacts; using inferred FEATURE_COLS: ['CategoryName', 'cur_mean', 'cur_std', 'coverage', 'volatility_ratio', 'intra_growth', 'month', 'week_of_year', 'day_of_week']


,CategoryName,probability_up,growth_signal,window_start,window_end
2,Storage,0.825835,1,2017-10-06,2017-11-02
1,Mother Board,0.820398,1,2017-10-06,2017-11-02
0,CPU,0.775864,1,2017-10-06,2017-11-02
3,Video Card,0.418398,1,2017-10-06,2017-11-02


## 5) Compute product share per category (last 90 days)
This gives the proportional allocation weights for each product within each category.

In [17]:
def compute_product_shares(df: pd.DataFrame, days: int = 90) -> pd.DataFrame:
    latest_date = df["OrderDate"].max().normalize()
    start = latest_date - pd.Timedelta(days=days-1)

    hist = df[(df["OrderDate"] >= start) & (df["OrderDate"] <= latest_date)].copy()

    prod_qty = (hist.groupby(["CategoryName","ProductName"])["OrderItemQuantity"]
                .sum()
                .reset_index()
                .rename(columns={"OrderItemQuantity":"product_qty"}))

    cat_qty = (prod_qty.groupby("CategoryName")["product_qty"]
               .sum()
               .reset_index()
               .rename(columns={"product_qty":"category_qty"}))

    out = prod_qty.merge(cat_qty, on="CategoryName", how="left")
    out["product_share"] = out["product_qty"] / (out["category_qty"] + 1e-9)
    return out, start, latest_date

shares, share_start, share_end = compute_product_shares(df_raw, SHARE_WINDOW_DAYS)
print("Share window:", share_start.date(), "→", share_end.date())
shares.sort_values(["CategoryName","product_share"], ascending=[True, False]).head(10)

Share window: 2017-08-05 → 2017-11-02


,CategoryName,ProductName,product_qty,category_qty,product_share
5,CPU,Intel Xeon E5-2695 V3 (OEM/Tray),148,698,0.212034
4,CPU,Intel Xeon E5-2683 V4,145,698,0.207736
1,CPU,Intel Xeon E5-2643 V3 (OEM/Tray),139,698,0.199140
3,CPU,Intel Xeon E5-2660 V4,118,698,0.169054
2,CPU,Intel Xeon E5-2650 V3 (OEM/Tray),104,698,0.148997
0,CPU,Intel Core i7-5960X (OEM/Tray),44,698,0.063037
6,Mother Board,ASRock C2750D4I,133,495,0.268687
7,Mother Board,ASRock X99 Extreme11,90,495,0.181818
10,Mother Board,Intel DG43RK,78,495,0.157576
8,Mother Board,Asus ROG STRIX X99 GAMING,73,495,0.147475


## 6) Allocate category forecast budget to products
We define a simple budget:

- `category_base_forecast = cur_mean × FORECAST_HORIZON_DAYS`
- `category_adjusted_forecast = category_base_forecast × (1 + ALPHA × probability_up)`

Then allocate:

- `recommended_qty = category_adjusted_forecast × product_share`

*Can tune `ALPHA` later if needed.*

In [18]:
# Category budget
latest_feat["category_base_forecast"] = latest_feat["cur_mean"] * FORECAST_HORIZON_DAYS
latest_feat["category_adjusted_forecast"] = latest_feat["category_base_forecast"] * (1 + ALPHA * latest_feat["probability_up"])

# Merge budget with shares
alloc = shares.merge(
    latest_feat[["CategoryName","probability_up","growth_signal","category_adjusted_forecast"]],
    on="CategoryName",
    how="left"
)

alloc["recommended_qty"] = (
    alloc["category_adjusted_forecast"] * alloc["product_share"]
).round(0).fillna(0).astype("Int64")

# Sort for readability
alloc_out = alloc[[
    "CategoryName","ProductName",
    "product_share","product_qty","category_qty",
    "probability_up","growth_signal",
    "category_adjusted_forecast","recommended_qty"
]].sort_values(["CategoryName","recommended_qty"], ascending=[True, False])

alloc_out.head(20)

,CategoryName,ProductName,product_share,product_qty,category_qty,probability_up,growth_signal,category_adjusted_forecast,recommended_qty
5,CPU,Intel Xeon E5-2695 V3 (OEM/Tray),0.212034,148,698,0.775864,1.0,201.250170,43
4,CPU,Intel Xeon E5-2683 V4,0.207736,145,698,0.775864,1.0,201.250170,42
1,CPU,Intel Xeon E5-2643 V3 (OEM/Tray),0.199140,139,698,0.775864,1.0,201.250170,40
3,CPU,Intel Xeon E5-2660 V4,0.169054,118,698,0.775864,1.0,201.250170,34
2,CPU,Intel Xeon E5-2650 V3 (OEM/Tray),0.148997,104,698,0.775864,1.0,201.250170,30
0,CPU,Intel Core i7-5960X (OEM/Tray),0.063037,44,698,0.775864,1.0,201.250170,13
6,Mother Board,ASRock C2750D4I,0.268687,133,495,0.820398,1.0,109.995527,30
7,Mother Board,ASRock X99 Extreme11,0.181818,90,495,0.820398,1.0,109.995527,20
10,Mother Board,Intel DG43RK,0.157576,78,495,0.820398,1.0,109.995527,17
8,Mother Board,Asus ROG STRIX X99 GAMING,0.147475,73,495,0.820398,1.0,109.995527,16


## 7) Export outputs (CSV)
Exports:
- `subsystem1_category_output_v2.csv`
- `subsystem1_product_recommendations_v2.csv`

These are versioned so the refreshed allocation output is easy to distinguish from older files.


In [19]:
# Normalize recommended quantity
alloc_out["recommended_qty_norm"] = (
    alloc_out["recommended_qty"] - alloc_out["recommended_qty"].min()
) / (
    alloc_out["recommended_qty"].max() - alloc_out["recommended_qty"].min() + 1e-6
)

# Create forecast score
alloc_out["forecast_score"] = (
    0.5 * alloc_out["probability_up"] +
    0.2 * alloc_out["product_share"] +
    0.3 * alloc_out["recommended_qty_norm"]
).clip(0, 1)

# Base reorder quantity
alloc_out["base_reorder_qty"] = alloc_out["recommended_qty"]
alloc_out.drop(columns=["recommended_qty"], inplace=True)

# Forecast confidence
alloc_out["confidence_score"] = alloc_out["probability_up"]

In [20]:
alloc_out = alloc_out[[
    "CategoryName",
    "ProductName",
    "product_share",
    "product_qty",
    "category_qty",
    "probability_up",
    "growth_signal",
    "category_adjusted_forecast",
    "base_reorder_qty",
    "confidence_score",
    "forecast_score"
]].sort_values("probability_up", ascending=False)

In [21]:
category_out = latest_feat[[
    "CategoryName", "probability_up", "growth_signal",
    "category_base_forecast", "category_adjusted_forecast",
    "window_start", "window_end"
]].sort_values("probability_up", ascending=False)

# Keep only products whose categories exist in category forecast output
valid_categories = set(category_out["CategoryName"])
alloc_out = alloc_out[alloc_out["CategoryName"].isin(valid_categories)].copy()

category_output_path = "subsystem1_category_output_v2.csv"
product_output_path = "subsystem1_product_recommendations_v2.csv"

category_out.to_csv(category_output_path, index=False)
alloc_out.to_csv(product_output_path, index=False)

print("Saved:")
print(" -", category_output_path)
print(" -", product_output_path)


Saved:
 - subsystem1_category_output_v2.csv
 - subsystem1_product_recommendations_v2.csv


### Do recommendations sum back to category budget?
### Are there categories with missing probability (should not happen)?

In [22]:
# Check sums close to category budget
check = (alloc_out.groupby("CategoryName")["base_reorder_qty"].sum()
         .reset_index()
         .merge(category_out[["CategoryName","category_adjusted_forecast"]], on="CategoryName", how="left"))

check["diff"] = check["base_reorder_qty"] - (
    check["category_adjusted_forecast"]
    .fillna(0)
    .round(0)
    .astype("Int64")
)
check.sort_values("diff", ascending=False).head(10)

,CategoryName,base_reorder_qty,category_adjusted_forecast,diff
0,CPU,202,201.250170,1
1,Mother Board,110,109.995527,0
3,Video Card,394,394.198864,0
2,Storage,169,169.550078,-1
